# 03 · Index & retrieve
Step 3: embed the chunks into Chroma, build the bm25s keyword index, and test dense / BM25 / hybrid retrieval.

Section 1 calls the OpenAI embeddings API (~1.2M tokens for both strategies, under $0.03). Sections 2–4 only read the saved indexes (queries are embedded, fractions of a cent).

Restart the kernel after editing anything in `src/`.

In [ ]:
import sys, pathlib
print(sys.executable)
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import json, pandas as pd
from IPython.display import Markdown, display
from finsight.index.vectorstore import build_all, MANIFEST, check_manifest
from finsight.index.bm25 import build_bm25
from finsight.retrieve.hybrid import search

## 1. Build the indexes
Chroma collections `fixed` and `semantic` are rebuilt into an empty `index/chroma_db/` every run (in-place rebuilds leave orphaned data and bloat the folder). Vectors come from the embedding cache, so only changed chunk text calls the API. bm25s indexes go to `index/bm25/`.

**Restart the kernel before running this cell**, so no Chroma client holds the folder open.

In [ ]:
counts = build_all()  # fresh index/chroma_db/ every time; vectors come from the embedding cache
for strategy in ['fixed', 'semantic']:
    build_bm25(strategy)
print('vectors:', counts)
check_manifest()
print(MANIFEST.read_text())

In [ ]:
questions = [
    ('What was AWS operating income in 2024?', 'Amazon', 2024),
    ('How much revenue did Google Cloud generate in 2024?', 'Alphabet', 2024),
    ('What was Intelligent Cloud revenue in fiscal year 2024?', 'Microsoft', 2024),
    ('What was free cash flow in 2024?', 'Amazon', 2024),
    ('What are the main risks related to AI regulation?', 'Alphabet', 2025),
    ('How many employees does Microsoft have?', 'Microsoft', 2025),
]

def show(q, company=None, year=None, strategy='fixed', k=5):
    rows = []
    for mode in ['dense', 'bm25', 'hybrid']:
        for rank, d in enumerate(search(q, strategy, mode, company=company, fiscal_year=year, k=k), 1):
            m = d.metadata
            rows.append({'mode': mode, 'rank': rank, 'chunk_id': m['chunk_id'], 'kind': m['kind'],
                         'section': m['section'][:40], 'page': m['page'] or '-', 'title': m['title'][:50]})
    print(f'Q: {q}   [{company} {year}, {strategy}]')
    display(pd.DataFrame(rows).pivot(index='rank', columns='mode', values='chunk_id'))
    return pd.DataFrame(rows)

res = show(*questions[0])

In [ ]:
res[res['mode'] == 'hybrid']

### Read the top hybrid hit
This is the text the LLM would see.

In [ ]:
top = search(questions[0][0], 'fixed', 'hybrid', company=questions[0][1], fiscal_year=questions[0][2], k=1)[0]
print(top.metadata)
display(Markdown(top.page_content[:3000]))

## 3. All test questions (hybrid, fixed)

In [ ]:
for q, company, year in questions[1:]:
    r = show(q, company, year)
    top = search(q, 'fixed', 'hybrid', company=company, fiscal_year=year, k=1)[0]
    print('TOP:', top.metadata['chunk_id'], '|', top.page_content[:300].replace(chr(10), ' '))
    print()

## 4. Same question, fixed vs semantic

In [ ]:
for strategy in ['fixed', 'semantic']:
    hits = search(questions[4][0], strategy, 'hybrid', company=questions[4][1], fiscal_year=questions[4][2], k=3)
    print(strategy)
    for d in hits:
        print('   ', d.metadata['chunk_id'], d.metadata['section'][:50], '|', d.page_content.split(chr(10), 1)[1][:150].replace(chr(10), ' '))

## Notes / issues found
_(fill in after reviewing)_